# 01 - Reimplementação da pipeline medalhão

Este notebook reproduz em **Python e pandas** a pipeline Bronze -> Silver -> Gold da Fase 2. Entram apenas os cinco CSVs originais do Indicador Criança Alfabetizada. As fontes externas são opcionais no enunciado e ficaram fora.

> A pipeline de dados não trata leakage. Ela preserva os dados. A seleção de variáveis acontece somente na EDA e na modelagem.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.preprocessing import run_bronze, run_silver, run_gold, ler_particionado
from src.preprocessing.config import ARQUIVOS_INEP, DADOS_DIR, LAKE_DIR
pd.set_option("display.max_columns", 30)
pd.DataFrame([
    {"entidade": entidade, "arquivo": arquivo,
     "tamanho_kb": round((DADOS_DIR / arquivo).stat().st_size / 1024, 1)}
    for entidade, arquivo in ARQUIVOS_INEP.items()
])

,entidade,arquivo,tamanho_kb
0,indicador_municipio,br_inep_avaliacao_alfabetizacao_municipio.csv,1385.8
1,indicador_uf,br_inep_avaliacao_alfabetizacao_uf.csv,8.2
2,meta_brasil,br_inep_avaliacao_alfabetizacao_meta_alfabetiz...,0.4
3,meta_uf,br_inep_avaliacao_alfabetizacao_meta_alfabetiz...,3.3
4,meta_municipio,br_inep_avaliacao_alfabetizacao_meta_alfabetiz...,752.6


## Bronze - ingestão fiel

A Bronze aplica tipos explícitos, mantém códigos como texto e adiciona hash da chave de negócio e linhagem. Nenhuma feature é removida.

In [2]:
resumo_bronze = run_bronze()
resumo_bronze

,entidade,registros,colunas
0,indicador_municipio,23995,19
1,indicador_uf,145,19
2,meta_brasil,3,15
3,meta_uf,54,16
4,meta_municipio,10704,17


## Silver - qualidade e padronização

A Silver deduplica por hash, traduz o código da rede, arredonda indicadores e separa registros inválidos.

In [3]:
resumo_silver = run_silver()
resumo_silver

,entidade,pass,quarentena
0,indicador_municipio,23995,0
1,indicador_uf,145,0
2,meta_brasil,3,0
3,meta_uf,54,0
4,meta_municipio,10704,0


## Gold - quatro visões analíticas

A Gold reproduz as quatro saídas da Fase 2. A principal é alfabetizacao_por_municipio, no grão município-ano-rede municipal. O dataset de ML ainda não é criado aqui.

In [4]:
resumo_gold = run_gold()
resumo_gold

,visao,registros,colunas
0,alfabetizacao_por_municipio,10896,11
1,evolucao_temporal,49,8
2,ranking_municipios,10896,8
3,comparacao_metas_nacionais,1,9


In [5]:
gold = ler_particionado(LAKE_DIR / "gold" / "alfabetizacao_por_municipio")
print(f"Gold principal: {gold.shape[0]:,} linhas x {gold.shape[1]} colunas")
display(gold.head())
display(gold.groupby("ano").agg(
    municipios=("id_municipio", "nunique"),
    taxa_media=("taxa_alfabetizacao", "mean")
).round(2))

Gold principal: 10,896 linhas x 13 colunas


,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,meta_2025,meta_2030,nivel_alfabetizacao,gap_meta_2025,status_meta_2025,_gold_processed_at,_ingestion_date,ano
0,1100031,2,municipal,69.10,767.88,72.53,80.0,3,-3.43,NAO_ATINGIU,2026-09-09T18:38:15.697007+00:00,2026-09-09,2023
1,1100072,2,municipal,58.20,747.89,65.31,80.0,2,-7.11,NAO_ATINGIU,2026-09-09T18:38:15.697007+00:00,2026-09-09,2023
2,1101609,2,municipal,50.70,745.68,60.25,80.0,2,-9.55,NAO_ATINGIU,2026-09-09T18:38:15.697007+00:00,2026-09-09,2023
3,1101807,2,municipal,55.69,752.37,63.63,80.0,2,-7.94,NAO_ATINGIU,2026-09-09T18:38:15.697007+00:00,2026-09-09,2023
4,1302900,2,municipal,53.17,731.07,61.93,80.0,2,-8.76,NAO_ATINGIU,2026-09-09T18:38:15.697007+00:00,2026-09-09,2023


,municipios,taxa_media
ano,,
2023,5448,60.28
2024,5448,62.80


## Diferenças de reimplementação

- AWS Glue, Spark, S3 e Athena foram substituídos por pandas e Parquet local.
- O particionamento ano=YYYY e a separação Bronze/Silver/Gold foram mantidos.
- As cinco fontes e as quatro visões Gold são as mesmas.
- As fontes externas e a Gold enriquecida da versão anterior deixaram de participar.
- Leakage não é responsabilidade desta etapa; será demonstrado e tratado no notebook 02.